In [11]:
import os
import sys
import numpy as np

from SciExpeM_API.SciExpeM import SciExpeM
from SciExpeM_API.Models import *

sciexp_scripts_path = r"/Users/lpratalimaffei/Library/CloudStorage/OneDrive-PolitecnicodiMilano/Luna/Universita/RICERCA/SCIEXPEM/SciExpeM_API/Examples/scripts"
if sciexp_scripts_path not in sys.path:
    sys.path.append(sciexp_scripts_path)

import build_sciexp_objects
import extract_data

my_sciexpem = SciExpeM(username='YOUR_USERNAME',
                       password='YOUR_PASSWORD', secure=False, port=8080)

CONVERT_TO_BAR = {'atm': 1.01325, 'bar': 1., 'Torr': 0.00133322, 'Pa': 1e-5}

Attention. SciExpeM is a singleton.


### FilePaper
 - description*
 - reference_doi*
 - author*
 - title*
 - year*
 - volume
 - page
 - journal

In [12]:
file_paper = FilePaper(reference_doi="10.1016/j.combustflame.2023.113046",
                       author="Meziane, I.; Delort, N.; Herbinet, O.; Bounaceur, R.; Battin-Leclerc, F.",
                       title="A comparative study of the oxidation of toluene and the three isomers of xylene.",
                       year=2023,
                       #journal="Combustion and Flame",
                       description="A comparative study of the oxidation of toluene and the three isomers of xylene. Combustion and Flame, 257, 113046. https://doi.org/10.1016/J.COMBUSTFLAME.2023.113046"
            )

### OpenSMOKE input file if available

In [13]:
datapath = os.path.join(sciexp_scripts_path, 'data')
osinputname = 'JSR_inputOS.dic'
# WARNING LIST OF SPECIES MUST HAVE THE NAMES OF THE MECH YOU ARE GOING TO SIMULATE - OTHERWISE, NEED REPLACEMENT
inputstr, extrainfo = extract_data.process_osinput(datapath, osinputname, profiles=False, flameinfo=True)  
# LA LISTA DI OUTPUTSPECIES DEVE AVERE SPECIE NEL MECCANISMO CHE SIMULERAI

In [14]:
# needed because XYLENE is lumped in CRECK. we need to assign P-XYLENE as reactant
extrainfo['molefractions'][0] = ['P-XYLENE', 'O2', 'HE']

In [15]:
print(extrainfo)

{'profileinfo': {}, 'commonprop': {'residence time': ['2.0', 's'], 'pressure': ['1.066', 'bar'], 'parametric temperature': ['750', '1100', 'K']}, 'character': {}, 'molefractions': [['P-XYLENE', 'O2', 'HE'], ['0.005', '0.105', '0.89']], 'oxidizermolefractions': []}


### Common properties

- name
- units
- value
- source_type

In [16]:
sourcetype = 'reported'
#######################
commonprop = []
for name, values in extrainfo['commonprop'].items():
    ci = CommonProperty(name=name, units=values[1], value=values[0], source_type=sourcetype)
    commonprop.append(ci)
    
# ADD OTHER COMMON PROPERTIES
#ci = CommonProperty(name='temperature', units='K', value='1120', source_type='reported')
#commonprop.append(ci)
############# DO NOT EDIT
# extract pressure
if 'pressure' in extrainfo['commonprop'].keys():
    Pval, Punit = extrainfo['commonprop']['pressure']
    P = float(Pval) * CONVERT_TO_BAR[Punit]


In [17]:
for c in commonprop:
    print(c.serialize())

{'name': 'residence time', 'units': 's', 'value': '2.0', 'source_type': 'reported'}
{'name': 'pressure', 'units': 'bar', 'value': '1.066', 'source_type': 'reported'}
{'name': 'parametric temperature', 'units': '1100', 'value': '750', 'source_type': 'reported'}


### Initial Species

- name
- units
- value
- source_type
- configuration

In [18]:
# THIS REFERS TO THE SIMULATION
# PREMIXED IS DEFAULT AND MUST BE INDICATED UNLESS IT'S A CF FLAME
comp_unit = 'mole fraction'
srctype = 'reported'
config = 'premixed'
# do not edit
################# do not edit
inspecies, fuelobjs = build_sciexp_objects.build_initial_species(
    my_sciexpem=my_sciexpem,
    molefractions=extrainfo['molefractions'],
    source_type=srctype,
    units=comp_unit,
    configuration=config,
)


Species: ['P-XYLENE', 'O2', 'HE']
Fuels: ['P-XYLENE']
Mole Frac: ['0.005', '0.105', '0.89']
config: ['premixed', 'premixed', 'premixed']


### Data columns

- name
- label (not comuplsory)
- species_object
- units
- data
- dg_id 
- dg_label
- source_type


In [19]:
sp_table = {
    'PHTHALANE': 'PHTHALAN',
    'BZUFUR': 'BZFUR'
}

# if not needed : empty dictionary

In [10]:
# read data
datafile = os.path.join(sciexp_scripts_path, 'data', 'JSR_data.txt')
df_data = extract_data.readdata(datafile, delzero=True)
# process data
datacols = []
srctype = 'digitized'
label = 'experimental_data'
#x = ['temperature', 'K']
#x = ['time', 's'] # NB must be 'residence time' for concentration time profile, but only works with 'time' lol
#y = ['concentration', 'mol/cm3']
x = ['temperature', 'K']
y = ['composition', 'mole fraction']
list_exclude_species = []
uncert_x = [] #currently unavailable
uncert_y = [0.1, 'relative']
#################### DO NOT EDIT################
TINF, TSUP = extract_data.temperature_bounds(
    extrainfo,
    x=x,
    df_data=df_data,
)
print('T max and min: {} - {} K'.format(TINF, TSUP))
for dg, df in df_data.items():
    # x axis
    x_data = list(df.index)
    x_datacol = DataColumn(name=x[0], units=x[1], dg_id=dg, dg_label=label, data=x_data,
                           source_type=srctype)
datacols = build_sciexp_objects.build_data_columns(
    df_data=df_data,
    my_sciexpem=my_sciexpem,
    x=x,
    y=y,
    source_type=srctype,
    label=label,
    sp_table=sp_table,
    list_exclude_species=list_exclude_species,
    uncert_x=uncert_x,
    uncert_y=uncert_y,
)

T max and min: 750.0 - 1100.0 K
P-XYLENE [<Species (136)>]
C7H8 [<Species (50)>]
C6H5C2H5 [<Species (47)>]
species P-CH3C6H4CHO not as preferred name: alternative names searched
P-CH3C6H4CHO [<Species (579)>]
O2 [<Species (2)>]
CO2 [<Species (10)>]
CO [<Species (4)>]
CH4 [<Species (6)>]
C2H4 [<Species (53)>]
C6H6 [<Species (39)>]
C6H5CHO [<Species (48)>]
P-CH3C6H4C2H3 [<Species (607)>]
C2H6 [<Species (13)>]
C3H6 [<Species (55)>]
C4H6 [<Species (62)>]
C6H5OH [<Species (49)>]
C3H4-A [<Species (15)>]
FURAN [<Species (535)>]
CH3CHO [<Species (69)>]
BZFUR [<Species (18)>]
C2H3CHO [<Species (52)>]
CH3COCH3 [<Species (256)>]
IC3H5CHO [<Species (461)>]
CUMENE [<Species (578)>]
C5H6 [<Species (64)>]
species LC5H8 not as preferred name: alternative names searched
LC5H8 [<Species (550)>]
CYC6H8 [<Species (229)>]
species P-CH3C6H4C2H5 not as preferred name: alternative names searched
P-CH3C6H4C2H5 [<Species (485)>]
species C9H8 not as preferred name: alternative names searched
C9H8 [<Species (35)>

### Assemble the experiment

In [55]:
PHI_INF = extract_data.equivalence_ratio(extrainfo['molefractions'])
PHI_SUP = PHI_INF

e = Experiment(reactor='stirred reactor', # [flow reactor, flame, stirred reactor ]
               experiment_type='jet stirred reactor measurement',
               file_paper=file_paper,
               reactor_modes=['premixed'], # da mettere in tutte le fiamme # coflow, couterflow, premixed, burner stabilized stagnation
               data_columns=datacols, 
               initial_species=inspecies, 
               common_properties=commonprop,
               os_input_file=inputstr,
               #t_inf = 1120, t_sup = 1120,
               t_inf = TINF, t_sup = TSUP,
               p_inf = P, p_sup = P,
               phi_inf=PHI_INF, phi_sup=PHI_SUP,
               # fuels=fuels,
               fuels_object = fuelobjs, # devi passargli gli id delle specie che sono fuels
               comment = 'unc is 5% for fuel, 10% for other species'
               #comment = 'technically not isothermal but simulated as such - T almost unvaried'
               #comment = 'C4H2 plot unclear, order might be 1e6 or 1e8 (person who digitized assumed 1e8); bath gas varies with experimental run (Ne, He), but should not affect the results'
               )

In [28]:
print(fuelobjs)

[<Species (136)>]


In [56]:
e.serialize()
# check that everything is ok

{'data_columns': [{'name': 'temperature',
   'units': 'K',
   'data': [600.0,
    625.0,
    650.0,
    675.0,
    700.0,
    725.0,
    750.0,
    775.0,
    800.0,
    825.0,
    850.0,
    875.0,
    900.0,
    925.0,
    950.0,
    975.0,
    1000.0,
    1020.0],
   'dg_id': 'dg1',
   'dg_label': 'experimental_data',
   'label': None,
   'source_type': 'digitized',
   'data_group_profile': None,
   'uncertainty_kind': None,
   'uncertainty_bound': None,
   'diz': None,
   'species_object': None,
   'uncertainty_reference': None},
  {'name': 'composition',
   'units': 'mole fraction',
   'data': [0.00464,
    0.00525,
    0.00495,
    0.00491,
    0.00499,
    0.00489,
    0.00502,
    0.00517,
    0.00517,
    0.00486,
    0.00462,
    0.00442,
    0.00404,
    0.00138,
    0.000497,
    0.000308,
    0.000107,
    6.41e-06],
   'dg_id': 'dg1',
   'dg_label': 'experimental_data',
   'label': None,
   'source_type': 'digitized',
   'data_group_profile': None,
   'uncertainty_kind': 

### Send Experiment

In [40]:
my_sciexpem.insertElement(e, verbose=True)

Experiment element inserted successfully.


In [ ]:
# aggiungi cella per verificare da sciexpem_API
# metti id di sciexp
e = my_sciexpem.filterDatabase(model_name='Experiment', id = )[0]
my_sciexpem.verifyExperiment(e, 'verified', verbose = True)

Experiment verified successfully.
